# 02 — Gestión de dispositivos e instancias

Este notebook cubre cómo descubrir, seleccionar y conectar dispositivos Ivium — incluyendo escenarios con múltiples instancias de IviumSoft ejecutándose simultáneamente.

### Conceptos

- **Instancia de IviumSoft**: una copia en ejecución de la aplicación IviumSoft. Pueden ejecutarse hasta 32 a la vez.
- **Dispositivo / canal**: el potenciostato físico conectado a una instancia dada.
- **Número de serie**: el identificador único impreso en cada unidad de hardware Ivium.

```
 IviumSoft instancia 1  ──►  CompactStat #A  (SN: 12345)
 IviumSoft instancia 2  ──►  CompactStat #B  (SN: 67890)
 IviumSoft instancia 3  ──►  (sin dispositivo)
```

Pyvium habla con **una instancia a la vez**. Se cambia entre ellas con `select_iviumsoft_instance()`.

> **Manejo de errores:** Todas las excepciones de Pyvium están documentadas en `01_getting_started.ipynb` — Sección 4. Este notebook usa `except Exception as e` en todos lados por brevedad.

In [ ]:
from pyvium import Pyvium

## 1. Descubrir instancias de IviumSoft en ejecución

In [ ]:
Pyvium.open_driver()

instances = Pyvium.get_active_iviumsoft_instances()
max_devices = Pyvium.get_max_device_number()

print(f"Active IviumSoft instances : {instances}")
print(f"Max manageable devices     : {max_devices}")

## 2. Seleccionar una instancia

Todas las llamadas subsiguientes operan sobre la instancia **actualmente seleccionada**. Si solo tienes un IviumSoft abierto, este paso es opcional — la instancia 1 está seleccionada por defecto.

In [ ]:
# Seleccionar instancia 1 (el primer IviumSoft en ejecución)
Pyvium.select_iviumsoft_instance(1)

status_code, status_label = Pyvium.get_device_status()
print(f"Instance 1 — status: {status_code} ('{status_label}')")

# Códigos de estado:
#  -1 : sin IviumSoft en este slot
#   0 : IviumSoft ejecutándose pero dispositivo no conectado
#   1 : dispositivo conectado e inactivo
#   2 : dispositivo conectado y ocupado (ejecutando un método)
#   3 : no se detectó dispositivo en USB

## 3. Iterar todas las instancias

Cuando se ejecutan múltiples dispositivos en paralelo, itera todas las instancias activas para obtener su estado.

In [ ]:
instances = Pyvium.get_active_iviumsoft_instances()

for instance_number in instances:
    Pyvium.select_iviumsoft_instance(instance_number)
    code, label = Pyvium.get_device_status()
    print(f"  Instance {instance_number}: {label}")

## 4. Conectar un dispositivo

"Conectar" le indica a IviumSoft que tome control del dispositivo físico. El dispositivo debe ser detectado en USB primero (estado != 3).

> **Efecto secundario:** `connect_device()` siempre restablece el **Auto E-ranging a desactivado** en el instrumento para establecer un estado conocido. Si dependes de que el Auto E-ranging esté activo, vuelve a activarlo explícitamente después de conectar con `set_bistat_mode(1)` (ver notebook `05_bipotentiostat_and_we32.ipynb`).

In [ ]:
Pyvium.select_iviumsoft_instance(1)

code, label = Pyvium.get_device_status()
print(f"Before connect: {label}")

if code == 0:  # IviumSoft ejecutándose pero ningún dispositivo conectado aún
    Pyvium.connect_device()
    code, label = Pyvium.get_device_status()
    print(f"After connect : {label}")
elif code == 1:
    print("Device already connected and idle")
elif code == 2:
    print("Device connected and busy")
elif code == 3:
    print("No device detected — check USB cable")

## 5. Leer el número de serie

El número de serie identifica de forma única cada dispositivo físico. Útil para confirmar qué dispositivo está activo cuando se usan múltiples unidades.

In [ ]:
serial = Pyvium.get_device_serial_number()
print(f"Device serial number: {serial}")

## 6. Seleccionar un dispositivo por número de serie

`select_serial_number()` es la forma recomendada de apuntar a un **dispositivo físico específico** independientemente de en qué slot de IviumSoft aparece. Esto es más robusto que depender de los números de slot, que pueden cambiar cuando los dispositivos se desconectan y reconectan.

Devuelve el índice base 0 del dispositivo en el menú desplegable de IviumSoft.

In [ ]:
TARGET_SERIAL = "12345"  # Reemplazar con el número de serie de tu dispositivo

index = Pyvium.select_serial_number(TARGET_SERIAL)
print(f"Device '{TARGET_SERIAL}' found at dropdown index {index}")

Pyvium.connect_device()
print("Device connected")

## 7. Multicanal: Seleccionar un canal

Ivium-n-Soft (la variante multicanal) gestiona múltiples canales dentro de una instancia de IviumSoft. `select_channel()` abre la pestaña y hace que ese canal esté activo.

Esto es **diferente de** `select_iviumsoft_instance()` — la selección de canal es dentro de una única ventana de IviumSoft, mientras que la selección de instancia cambia entre ventanas separadas de IviumSoft.

In [ ]:
CHANNEL = 1  # Los canales están indexados desde 1

Pyvium.select_channel(CHANNEL)
print(f"Channel {CHANNEL} selected")

Pyvium.connect_device()
print("Device on channel connected")

## 8. Desconectar

Llama a `disconnect_device()` para liberar el control de IviumSoft sobre el dispositivo sin cerrar IviumSoft. Útil en flujos de trabajo con múltiples dispositivos donde la propiedad necesita transferirse.

In [ ]:
Pyvium.disconnect_device()
print("Device disconnected")

code, label = Pyvium.get_device_status()
print(f"Status after disconnect: {label}")

## 9. Flujo de trabajo completo con múltiples instancias

Plantilla para iterar todas las instancias activas, conectar cada una, hacer algo y luego pasar a la siguiente.

In [ ]:
import time

try:
    Pyvium.open_driver()
    instances = Pyvium.get_active_iviumsoft_instances()
    print(f"Found {len(instances)} active instance(s): {instances}")

    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        code, label = Pyvium.get_device_status()
        print(f"\nInstance {inst}: {label}")

        if code == 0:
            Pyvium.connect_device()
            serial = Pyvium.get_device_serial_number()
            print(f"  Connected — serial: {serial}")

            # ... realizar trabajo en este dispositivo ...

            Pyvium.disconnect_device()
            print(f"  Disconnected")

        elif code == 1:
            serial = Pyvium.get_device_serial_number()
            print(f"  Already connected — serial: {serial}")

        elif code == 2:
            print(f"  Busy — skipping")

        elif code == 3:
            print(f"  No device on USB — skipping")

finally:
    Pyvium.close_driver()

---

## Resumen

| Tarea | Método |
|------|-------|
| Listar ventanas de IviumSoft en ejecución | `get_active_iviumsoft_instances()` |
| Cambiar entre ventanas de IviumSoft | `select_iviumsoft_instance(n)` |
| Obtener estado del dispositivo | `get_device_status()` → `(code, label)` |
| Conectar el dispositivo seleccionado | `connect_device()` |
| Desconectar dispositivo | `disconnect_device()` |
| Obtener número de serie | `get_device_serial_number()` |
| Seleccionar dispositivo por número de serie | `select_serial_number(sn)` |
| Seleccionar pestaña multicanal | `select_channel(n)` |

## Siguiente

- **`03_direct_mode_basics.ipynb`** — Establecer potencial, leer corriente, controlar la celda directamente